In [ ]:
"""Script de análisis de resultados GGS.

Lee el CSV de resultados más reciente para un modelo dado y muestra:
    - Resumen general (total configs, éxitos, fallos)
    - Distribución de métricas
    - Top-N configuraciones
    - Análisis de importancia de hiperparámetros (media por valor)

Run desde el root del proyecto:
    python analyze_ggs_results.py --model nb
    python analyze_ggs_results.py --model svr
    python analyze_ggs_results.py --file outputs/ggs/results/MiModelo_hash_ts.csv
"""

import argparse
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Mapeo modelo → nombre de clase
# ---------------------------------------------------------------------------

_MODEL_NAMES = {
    'nb':  'NegativeBinomialPiecewise',
    'svr': 'SVRModel',
}

_METRICS = ['mean_S_score', 'mean_C_index', 'mean_MAE', 'mean_RMSE']


def _parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description='Análisis de resultados GGS',
    )
    group = parser.add_mutually_exclusive_group(required=True)
    group.add_argument(
        '--model', choices=list(_MODEL_NAMES.keys()),
        help='Modelo a analizar (nb, svr)',
    )
    group.add_argument(
        '--file', type=str,
        help='Ruta directa al CSV de resultados',
    )
    parser.add_argument(
        '--top', type=int, default=10,
        help='Número de top configs a mostrar (default: 10)',
    )
    return parser.parse_args()


def _find_latest_results(model_name: str) -> Path:
    """Encuentra el CSV de resultados más reciente para el modelo."""
    results_dir = Path('outputs/ggs/results')
    pattern = f'{model_name}_*.csv'
    files = sorted(results_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"No se encontraron resultados para {model_name} en {results_dir}"
        )
    return files[-1]  # más reciente por timestamp


def _sep(title: str, width: int = 60) -> None:
    print(f"\n{'='*width}")
    print(f"  {title}")
    print(f"{'='*width}")


def _fmt(val: float, decimals: int = 4) -> str:
    return f"{val:.{decimals}f}" if not np.isnan(val) else "NaN"


def analyze(results_path: Path, top_n: int) -> None:
    df = pd.read_csv(results_path)

    # Identificar columnas de hiperparámetros
    non_param_cols = set(_METRICS) | {'Success'}
    param_cols = [c for c in df.columns if c not in non_param_cols]

    # Verificar que las columnas de métricas existen
    missing = [m for m in _METRICS if m not in df.columns]
    if missing:
        raise ValueError(
            f"Columnas de métricas no encontradas en el CSV: {missing}\n"
            f"Columnas disponibles: {df.columns.tolist()}"
        )

    # Añadir Success
    df['Success'] = df['mean_S_score'].apply(
        lambda x: 0 if pd.isna(x) else 1
    )

    n_total   = len(df)
    n_success = int(df['Success'].sum())
    n_failed  = n_total - n_success

    # -----------------------------------------------------------------------
    # Sección 1 — Resumen general
    # -----------------------------------------------------------------------
    _sep("1 — Resumen general")
    print(f"  Archivo:          {results_path.name}")
    print(f"  Total configs:    {n_total}")
    print(f"  Exitosas:         {n_success} ({100*n_success/n_total:.1f}%)")
    print(f"  Fallidas (NaN):   {n_failed} ({100*n_failed/n_total:.1f}%)")
    print(f"  Hiperparámetros:  {param_cols}")

    # -----------------------------------------------------------------------
    # Sección 2 — Distribución de métricas (configs exitosas)
    # -----------------------------------------------------------------------
    _sep("2 — Distribución de métricas (configs exitosas)")

    df_ok = df[df['Success'] == 1].copy()

    if len(df_ok) == 0:
        print("  No hay configs exitosas.")
    else:
        stats = df_ok[_METRICS].agg(['mean', 'std', 'min', 'median', 'max'])
        print(f"\n  {'Métrica':<18} {'Media':>10} {'Std':>10} "
              f"{'Min':>10} {'Mediana':>10} {'Max':>10}")
        print(f"  {'-'*58}")
        for metric in _METRICS:
            row = stats[metric]
            print(f"  {metric:<18} {_fmt(row['mean']):>10} {_fmt(row['std']):>10} "
                  f"{_fmt(row['min']):>10} {_fmt(row['median']):>10} "
                  f"{_fmt(row['max']):>10}")

    # -----------------------------------------------------------------------
    # Sección 3 — Top-N configuraciones
    # -----------------------------------------------------------------------
    _sep(f"3 — Top {top_n} configuraciones (por S-Score)")

    df_sorted = df_ok.sort_values('mean_S_score', ascending=True).head(top_n)
    display_cols = param_cols + _METRICS
    print(df_sorted[display_cols].to_string(index=False))

    # -----------------------------------------------------------------------
    # Sección 4 — Análisis por hiperparámetro
    # -----------------------------------------------------------------------
    _sep("4 — Media de métricas por valor de hiperparámetro")

    for param in param_cols:
        n_unique = df_ok[param].nunique()
        if n_unique <= 1 or n_unique > 20:
            continue  # saltar si no aporta info

        print(f"\n  [ {param} ]")
        group = (
            df_ok.groupby(param)[_METRICS]
            .mean()
            .sort_values('mean_S_score')
        )
        print(group.round(4).to_string())

    # -----------------------------------------------------------------------
    # Sección 5 — Configuración ganadora
    # -----------------------------------------------------------------------
    _sep("5 — Configuración ganadora")

    best = df_sorted.iloc[0]
    print()
    for col in param_cols:
        print(f"  {col:<22} = {best[col]}")
    print()
    for metric in _METRICS:
        print(f"  {metric:<22} = {_fmt(best[metric])}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main() -> None:
    results_path = Path('outputs/ggs/results/NegativeBinomialPiecewise_13f6c147_20260507_2337.csv')
    print(f"\n  Analizando: {results_path}")
    analyze(results_path, top_n=10)
    print()


if __name__ == '__main__':
    main()